# RNN기반 분류기

In [1]:
# 데이터 로딩
from sklearn.datasets import fetch_20newsgroups

categories = ['comp.graphics', 'sci.space', 'rec.sport.baseball']
newsgroups = fetch_20newsgroups(subset='all', categories=categories)
X = newsgroups.data
y = newsgroups.target
print(newsgroups.target_names)
print(X[0])
print(y[0])

c:\Users\Playdata\AppData\Local\miniforge3\envs\dl_nlp_env\Lib\site-packages\sklearn\datasets\_base.py:1519: UserWarning: Retry downloading from url: https://ndownloader.figshare.com/files/5975967
  warnings.warn(f"Retry downloading from url: {remote.url}")


HTTPError: HTTP Error 403: Forbidden

In [2]:
import zipfile
import re
import numpy as np
from sklearn.utils import Bunch
from sklearn.utils import check_random_state

zip_path = "archive.zip"

categories = [
    "comp.graphics",
    "sci.space",
    "rec.sport.baseball"
]

data = []
target = []

with zipfile.ZipFile(zip_path, "r") as z:
    for label, name in enumerate(categories):
        filename = f"{name}.txt"

        text = z.read(filename).decode("latin1")

        # 각 문서는 보통 From: 으로 시작하므로 From: 기준으로 분리한다.
        docs = re.split(r"(?m)(?=^From: )", text)
        docs = [doc.strip() for doc in docs if doc.strip()]

        data.extend(docs)
        target.extend([label] * len(docs))

# fetch_20newsgroups()와 비슷하게 데이터를 섞는다.
rng = check_random_state(42)
indices = np.arange(len(data))
rng.shuffle(indices)

data = [data[i] for i in indices]
target = np.array([target[i] for i in indices])

newsgroups = Bunch(
    data=data,
    target=target,
    target_names=categories
)

X = newsgroups.data
y = newsgroups.target

print(newsgroups.target_names)
print(X[0])
print(y[0])

['comp.graphics', 'sci.space', 'rec.sport.baseball']
From: pyron@skndiv.dseg.ti.com (Dillon Pyron)
Subject: Re: A WRENCH in the works?


In article <25228@ksr.com>, jfw@ksr.com (John F. Woods) writes:
>nanderso@Endor.sim.es.com (Norman Anderson) writes:
>>jmcocker@eos.ncsu.edu (Mitch) writes:
>>>effect that one of the SSRBs that was recovered after the
>>>recent space shuttle launch was found to have a wrench of
>>>some sort rattling around apparently inside the case.
>>I heard a similar statement in our local news (UTAH) tonight. They referred
>>to the tool as "...the PLIERS that took a ride into space...". They also
>>said that a Thiokol (sp?) employee had reported missing a tool of some kind
>>during assembly of one SRB.

It was a test of the first reusable tool.

>
>I assume, then, that someone at Thiokol put on their "manager's hat" and said
>that pissing off the customer by delaying shipment of the SRB to look inside
>it was a bad idea, regardless of where that tool might have en

In [3]:
# 데이터전처리
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

vocab_size = 10000
max_len = 200

tokenizer = Tokenizer(num_words=vocab_size, oov_token='<OOV>')
tokenizer.fit_on_texts(X)
X_encoded = tokenizer.texts_to_sequences(X)
X_padded = pad_sequences(X_encoded, maxlen=max_len)
print(X_padded.shape)

(6489, 200)


In [4]:
# 데이터분리/텐서변환
import torch
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset, DataLoader

X_train, X_test, y_train, y_test = train_test_split(torch.tensor(X_padded), torch.tensor(y), test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

# dataset/dataloader
train_dataset = TensorDataset(X_train, y_train)
val_dataset = TensorDataset(X_val, y_val)
test_dataset = TensorDataset(X_test, y_test)

batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [5]:
# 모델 생성
import torch.nn as nn

class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.lstm = nn.LSTM(embedding_dim, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, 3)

    def forward(self, x):
        x = self.embedding(x)
        _, (h, c) = self.lstm(x)
        out = self.fc(h[-1])
        return out

In [6]:
# 모델 학습
import torch.optim as optim

embedding_dim = 100
hidden_size = 128

model = LSTMClassifier(vocab_size, embedding_dim, hidden_size)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 학습루프
train_losses, train_accs = [], []
val_losses, val_accs = [], []

epochs = 50
for epoch in range(epochs):

    # 학습
    model.train()
    train_loss, train_correct, train_total = 0, 0, 0

    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        output = model(X_batch)
        loss = criterion(output, y_batch)
        loss.backward()
        optimizer.step()

        train_loss += loss.detach().item()
        pred = output.argmax(dim=1)
        train_correct += (pred == y_batch).sum().detach().item()
        train_total += len(y_batch)

    train_loss /= len(train_loader)
    train_acc = train_correct / train_total
    train_losses.append(train_loss)
    train_accs.append(train_acc)

    # 검증
    model.eval()
    val_loss, val_correct, val_total = 0, 0, 0

    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            output = model(X_batch)
            loss = criterion(output, y_batch)

            val_loss += loss.detach().item()
            pred = output.argmax(dim=1)
            val_correct += (pred == y_batch).sum().detach().item()
            val_total += len(y_batch)

        val_loss /= len(val_loader)
        val_acc = val_correct / val_total
        val_losses.append(val_loss)
        val_accs.append(val_acc)

    # 출력(train_loss, val_loss)
    print(f'Epoch {epoch + 1}/{epochs}: '
          f'Train Loss {train_loss:.4f}, '
          f'Train Acc {train_acc:.4f}, '
          f'Val Loss {val_loss:.4f}, '
          f'Val Acc {val_acc:.4f}, ')

Epoch 1/50: Train Loss 0.2668, Train Acc 0.9465, Val Loss 0.0204, Val Acc 0.9962, 
Epoch 2/50: Train Loss 0.0144, Train Acc 0.9966, Val Loss 0.0137, Val Acc 0.9962, 
Epoch 3/50: Train Loss 0.0149, Train Acc 0.9959, Val Loss 0.0097, Val Acc 0.9962, 
Epoch 4/50: Train Loss 0.0047, Train Acc 0.9990, Val Loss 0.0052, Val Acc 0.9981, 
Epoch 5/50: Train Loss 0.0036, Train Acc 0.9990, Val Loss 0.0027, Val Acc 0.9990, 
Epoch 6/50: Train Loss 0.0012, Train Acc 0.9998, Val Loss 0.0022, Val Acc 0.9990, 
Epoch 7/50: Train Loss 0.0006, Train Acc 1.0000, Val Loss 0.0009, Val Acc 1.0000, 
Epoch 8/50: Train Loss 0.0004, Train Acc 1.0000, Val Loss 0.0011, Val Acc 1.0000, 
Epoch 9/50: Train Loss 0.0002, Train Acc 1.0000, Val Loss 0.0006, Val Acc 1.0000, 
Epoch 10/50: Train Loss 0.0002, Train Acc 1.0000, Val Loss 0.0005, Val Acc 1.0000, 
Epoch 11/50: Train Loss 0.0001, Train Acc 1.0000, Val Loss 0.0005, Val Acc 1.0000, 
Epoch 12/50: Train Loss 0.0001, Train Acc 1.0000, Val Loss 0.0002, Val Acc 1.0000, 
E

In [7]:
# 모델 평가
from sklearn.metrics import classification_report

model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        output = model(X_batch)
        loss = criterion(output, y_batch)
        pred = output.argmax(dim=1)

        all_preds.extend(pred.detach().numpy())
        all_labels.extend(y_batch.detach().numpy())

print(classification_report(all_labels, all_preds, target_names=newsgroups.target_names))

                    precision    recall  f1-score   support

     comp.graphics       1.00      1.00      1.00       524
         sci.space       1.00      1.00      1.00       387
rec.sport.baseball       1.00      1.00      1.00       387

          accuracy                           1.00      1298
         macro avg       1.00      1.00      1.00      1298
      weighted avg       1.00      1.00      1.00      1298



## 사전학습된 임베딩 적용하기

In [8]:
from gensim.models import FastText

fasttext_model = FastText.load('ted_en_fasttext.model')
print(fasttext_model.vector_size)

100


In [9]:
import numpy as np

embedding_dim = fasttext_model.vector_size
embedding_matrix = np.zeros((vocab_size, embedding_dim))

word_index = tokenizer.word_index # 38000
word_index = {word:index for word, index in word_index.items() if index < vocab_size}
print(len(word_index)) # 10000

for word, index in word_index.items():
    if word in fasttext_model.wv:
        embedding_matrix[index] = fasttext_model.wv[word]

9999


In [10]:
# 모델 생성
import torch.nn as nn

class LSTMClassifier2(nn.Module):
    def __init__(self, vocab_size, embedding_dim, embedding_matrix, hidden_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.embedding.weight.data.copy_(torch.from_numpy(embedding_matrix))
        self.embedding.weight.requires_grad = False # 사전학습된 임베딩은 추가학습 안함

        self.lstm = nn.LSTM(embedding_dim, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, 3)

    def forward(self, x):
        x = self.embedding(x)
        _, (h, c) = self.lstm(x)
        out = self.fc(h[-1])
        return out


In [11]:
# 모델 학습
import torch.optim as optim

embedding_dim = 100
hidden_size = 128

model = LSTMClassifier2(vocab_size, embedding_dim, embedding_matrix, hidden_size)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)

# 학습루프
train_losses, train_accs = [], []
val_losses, val_accs = [], []

epochs = 100
for epoch in range(epochs):

    # 학습
    model.train()
    train_loss, train_correct, train_total = 0, 0, 0

    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        output = model(X_batch)
        loss = criterion(output, y_batch)
        loss.backward()
        optimizer.step()

        train_loss += loss.detach().item()
        pred = output.argmax(dim=1)
        train_correct += (pred == y_batch).sum().detach().item()
        train_total += len(y_batch)

    train_loss /= len(train_loader)
    train_acc = train_correct / train_total
    train_losses.append(train_loss)
    train_accs.append(train_acc)

    # 검증
    model.eval()
    val_loss, val_correct, val_total = 0, 0, 0

    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            output = model(X_batch)
            loss = criterion(output, y_batch)

            val_loss += loss.detach().item()
            pred = output.argmax(dim=1)
            val_correct += (pred == y_batch).sum().detach().item()
            val_total += len(y_batch)

        val_loss /= len(val_loader)
        val_acc = val_correct / val_total
        val_losses.append(val_loss)
        val_accs.append(val_acc)

    # 출력(train_loss, val_loss)
    print(f'Epoch {epoch + 1}/{epochs}: '
          f'Train Loss {train_loss:.4f}, '
          f'Train Acc {train_acc:.4f}, '
          f'Val Loss {val_loss:.4f}, '
          f'Val Acc {val_acc:.4f}, ')

Epoch 1/100: Train Loss 1.0731, Train Acc 0.3764, Val Loss 1.0387, Val Acc 0.3792, 
Epoch 2/100: Train Loss 0.8512, Train Acc 0.6604, Val Loss 0.5761, Val Acc 0.9326, 
Epoch 3/100: Train Loss 0.4035, Train Acc 0.9523, Val Loss 0.2785, Val Acc 0.9500, 
Epoch 4/100: Train Loss 0.2294, Train Acc 0.9706, Val Loss 0.1736, Val Acc 0.9731, 
Epoch 5/100: Train Loss 0.1447, Train Acc 0.9817, Val Loss 0.1196, Val Acc 0.9894, 
Epoch 6/100: Train Loss 0.0992, Train Acc 0.9899, Val Loss 0.0862, Val Acc 0.9913, 
Epoch 7/100: Train Loss 0.0752, Train Acc 0.9908, Val Loss 0.0644, Val Acc 0.9962, 
Epoch 8/100: Train Loss 0.0566, Train Acc 0.9942, Val Loss 0.0481, Val Acc 0.9952, 
Epoch 9/100: Train Loss 0.0423, Train Acc 0.9954, Val Loss 0.0395, Val Acc 0.9962, 
Epoch 10/100: Train Loss 0.0387, Train Acc 0.9949, Val Loss 0.0398, Val Acc 0.9962, 
Epoch 11/100: Train Loss 0.0329, Train Acc 0.9952, Val Loss 0.0306, Val Acc 0.9962, 
Epoch 12/100: Train Loss 0.0273, Train Acc 0.9961, Val Loss 0.0271, Val Ac

In [12]:
# 모델 평가
from sklearn.metrics import classification_report

model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        output = model(X_batch)
        loss = criterion(output, y_batch)
        pred = output.argmax(dim=1)

        all_preds.extend(pred.detach().numpy())
        all_labels.extend(y_batch.detach().numpy())

print(classification_report(all_labels, all_preds, target_names=newsgroups.target_names))

                    precision    recall  f1-score   support

     comp.graphics       1.00      1.00      1.00       524
         sci.space       1.00      1.00      1.00       387
rec.sport.baseball       1.00      1.00      1.00       387

          accuracy                           1.00      1298
         macro avg       1.00      1.00      1.00      1298
      weighted avg       1.00      1.00      1.00      1298

